# 024-class-compatibility-matching

# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import TwoSlopeNorm
from matplotlib.figure import Figure
from tqdm.auto import tqdm

from uavloc.data.drone_frames import DroneFrame, frame_place, load_frames
from uavloc.data.match_pairs import PairBank, PairBankConfig, build_pair_bank, village_split
from uavloc.data.multidate import load_date
from uavloc.datasets import dataset_ref
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import CLASSES, SegmentationFeatures, load_segmenter
from uavloc.matching.benchmark import (HARM_MARGIN_M, benchmark_draws, median_ms_per_correction,
                                       paired_frame_bootstrap, score_draws, summarise)
from uavloc.matching.compatibility import estimate_pmi, register_pmi
from uavloc.matching.correlator import PositionCorrelator, weighted_coverage
from uavloc.track import REPO_ROOT, experiment_dir, log_run, set_seed

# Params

In [ ]:
EXP = "024-class-compatibility-matching"
NOTE = ""
LOG_RUN = True

MATCHING = "weighted-ncc"         # "weighted-ncc" (024.1, = 007.1) | "pmi" (024.2) | "mi" (024.3)
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)   # background, building, track, woody, field, water: control arm and blind rule
REFERENCE_RUN = "024.1"           # paired frame bootstrap of every other arm against this run's records
N_BOOTSTRAP = 10_000
CI_PCT = 95.0
BOOTSTRAP_SEED = 42

SEED = 42
DEVICE = "mps"
NUM_THREADS = 2                   # shared CPU
PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
FRAMES_DATASET = "village-crossdate-frames"
HOLDOUT_DATASET = "village-crossdate-holdout"
MAP_DATE = "newest"
WINDOW = "rect"
SEARCH_SCALE = 1.7
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 10.0
N_DIRECTIONS = 8
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0
SEG_ARCHITECTURE = "lraspp"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385

EXCLUDE_PLACES = ["birky-49p9105-23p9192", "bushcha-50p2422-25p8305", "dubrova-49p5175-24p0765"]
N_VAL_VILLAGES = 10
SPLIT_SEED = 42
DRONE_DATE = "current"
ALTITUDE_M = 40.0
MARGIN_M = 80.0
N_TRAIN_PAIRS = 20_000            # 023's train bank, reused from its cache
BANK_SEED = 42
PMI_SMOOTHING = 1e-4

TIMING_DEVICE = "cpu"
TIMING_FRAMES = 8
TIMING_REPEATS = 3

# Data

In [ ]:
set_seed(SEED)
torch.set_num_threads(NUM_THREADS)
artifacts = experiment_dir(EXP) / "artifacts"
artifacts.mkdir(exist_ok=True)

In [ ]:
train_villages, val_villages = village_split(EXCLUDE_PLACES, N_VAL_VILLAGES, SPLIT_SEED)
len(train_villages), val_villages

In [ ]:
def load_bench(dataset: str, places: list[str]) -> dict[str, list[DroneFrame]]:
    """Frames of `dataset` grouped by place; raises if a place has none."""
    frames = load_frames(dataset)
    bench = {place: [f for f in frames if frame_place(f) == place] for place in places}
    missing = [place for place, members in bench.items() if not members]
    if missing:
        raise ValueError(f"{dataset} has no frames for {missing}")
    return bench

In [ ]:
benches = {"": load_bench(FRAMES_DATASET, PLACES), "holdout_": load_bench(HOLDOUT_DATASET, val_villages)}
{prefix or "primary": {place: len(m) for place, m in bench.items()} for prefix, bench in benches.items()}

In [ ]:
maps = {place: load_date(place, MAP_DATE) for bench in benches.values() for place in bench}

In [ ]:
PAIR_CONFIG = PairBankConfig(drone_date=DRONE_DATE, altitude_m=ALTITUDE_M, fov_w_deg=FOV_W_DEG, margin_m=MARGIN_M,
                             search_scale=SEARCH_SCALE, match_gsd_m=MATCH_GSD_M,
                             sat_feature_crop_pct=SAT_FEATURE_CROP_PCT, seg_channels=SEG_CHANNELS,
                             seg_checkpoint=SEG_CHECKPOINT)

In [ ]:
RUN_CONFIG = {
    "method": "segmentation", "matching": MATCHING,
    "class_weights": CLASS_WEIGHTS if MATCHING == "weighted-ncc" else None,
    "noise_m": NOISE_M, "primary_noise_m": PRIMARY_NOISE_M, "n_directions": N_DIRECTIONS, "bench_seed": SEED,
    "window": WINDOW, "search_scale": SEARCH_SCALE, "search_radius_m": None, "prior_sigma_m": None, "gate": "none",
    "fov_w_deg": FOV_W_DEG, "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
    "seg_architecture": SEG_ARCHITECTURE, "seg_checkpoint_sat": SEG_CHECKPOINT, "seg_checkpoint_drone": SEG_CHECKPOINT,
    "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": MATCH_GSD_M, "match_gsd_m": MATCH_GSD_M,
    "frames_dataset": FRAMES_DATASET, "frames_dataset_ref": dataset_ref(FRAMES_DATASET),
    "holdout_dataset": HOLDOUT_DATASET, "holdout_dataset_ref": dataset_ref(HOLDOUT_DATASET),
    "places": PLACES, "map_date": MAP_DATE,
    "blind_rule": {"class_weights": CLASS_WEIGHTS, "blind": "weighted_coverage == 0"},
    "split": {"exclude": EXCLUDE_PLACES, "n_val": N_VAL_VILLAGES, "seed": SPLIT_SEED, "val": val_villages},
    "reference_run": REFERENCE_RUN if MATCHING != "weighted-ncc" else None,
    "bootstrap": {"n": N_BOOTSTRAP, "ci_pct": CI_PCT, "seed": BOOTSTRAP_SEED},
    "num_threads": NUM_THREADS, "timing_device": TIMING_DEVICE,
    "pmi": None,
}

# Model

In [ ]:
model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, SEG_ARCHITECTURE)

In [ ]:
def pmi_table(bank: PairBank) -> np.ndarray:
    """W counted on `bank`, cached in artifacts together with the bank and smoothing it was counted with."""
    weights_path, meta_path = artifacts / "pmi-W.npy", artifacts / "pmi-W.json"
    meta = {"bank": bank.root.name, "pairs": len(bank), "smoothing": PMI_SMOOTHING}
    if weights_path.exists() and meta_path.exists() and json.loads(meta_path.read_text()) == meta:
        return np.load(weights_path)
    weights = estimate_pmi(bank, PMI_SMOOTHING)
    np.save(weights_path, weights)
    meta_path.write_text(json.dumps(meta, indent=1))
    return weights

In [ ]:
pmi_bank = build_pair_bank(train_villages, N_TRAIN_PAIRS, PAIR_CONFIG, model, DEVICE, BANK_SEED) if MATCHING == "pmi" else None
W = pmi_table(pmi_bank) if pmi_bank is not None else None
if W is not None:
    register_pmi(W)
    RUN_CONFIG["pmi"] = {"W": W.round(6).tolist(), "smoothing": PMI_SMOOTHING, "bank": pmi_bank.root.name,
                         "pairs": len(pmi_bank), "villages": len(train_villages), "bank_seed": BANK_SEED,
                         "drone_date": DRONE_DATE, "altitude_m": ALTITUDE_M, "margin_m": MARGIN_M}
pd.DataFrame(W, index=CLASSES, columns=CLASSES).round(2) if W is not None else None

In [ ]:
def make_correlator(place: str, device: str, segmenter: torch.nn.Module) -> PositionCorrelator:
    """007.1's correlator on `place`, scored by `MATCHING`."""
    features = SegmentationFeatures(segmenter, segmenter, maps[place], SEG_CHANNELS, MATCH_GSD_M, MATCH_GSD_M, device)
    return PositionCorrelator(features, correlator_config(RUN_CONFIG))

In [ ]:
correlators = {place: make_correlator(place, DEVICE, model) for bench in benches.values() for place in bench}
next(iter(correlators.values())).config

# Run

In [ ]:
def bench_records(bench: dict[str, list[DroneFrame]], desc: str) -> pd.DataFrame:
    """Every frame x bearing x level of one bench, scored; the draws are identical for every arm (SEED)."""
    parts = []
    for place, members in tqdm(bench.items(), desc=desc):
        correlator = correlators[place]
        draws = [d for noise_m in tqdm(NOISE_M, desc=place.split("-")[0], leave=False)
                 for d in benchmark_draws(correlator, members, noise_m, N_DIRECTIONS, SEED)]
        parts.append(score_draws(draws, correlator.config).assign(place=place))
    records = pd.concat(parts, ignore_index=True)
    records["harm"] = records["error_m"] > records["prior_error_m"] + HARM_MARGIN_M
    return records

In [ ]:
records = {prefix: bench_records(bench, prefix or "primary") for prefix, bench in benches.items()}

In [ ]:
def bench_subsets(bench: dict[str, list[DroneFrame]]) -> dict[str, set[str]]:
    """all / covered / blind / per place. Blind: the query holds no class weighted by the hand weights."""
    names = {f.name for members in bench.values() for f in members}
    blind = {f.name for place, members in bench.items() for f in members
             if weighted_coverage(correlators[place].drone_map(f), CLASS_WEIGHTS) == 0}
    return {"all": names, "covered": names - blind, "blind": blind,
            **{place.split("-")[0]: {f.name for f in members} for place, members in bench.items()}}

In [ ]:
def run_metrics(records: pd.DataFrame, subsets: dict[str, set[str]], prefix: str) -> dict[str, float]:
    """007's metrics on every subset, keys prefixed per bench."""
    metrics = summarise(records, subsets, PRIMARY_NOISE_M)
    metrics["blind_frame_share"] = len(subsets["blind"]) / len(subsets["all"])
    return {f"{prefix}{key}": value for key, value in metrics.items()}

In [ ]:
def paired_vs_reference(records: pd.DataFrame, subsets: dict[str, set[str]], prefix: str) -> dict[str, float]:
    """Paired frame bootstrap of (this run - REFERENCE_RUN) at the primary level, on all / covered / blind."""
    reference = pd.read_csv(artifacts / f"{prefix}records-{REFERENCE_RUN}.csv")
    out = {}
    for name in ("all", "covered", "blind"):
        members = subsets[name]
        if not members:
            continue
        stats = paired_frame_bootstrap(records[records["frame"].isin(members)],
                                       reference[reference["frame"].isin(members)],
                                       PRIMARY_NOISE_M, N_BOOTSTRAP, CI_PCT, BOOTSTRAP_SEED)
        out.update({f"{prefix}paired_{key}_{name}": value for key, value in stats.items()})
    return out

In [ ]:
subsets = {prefix: bench_subsets(bench) for prefix, bench in benches.items()}
metrics = {key: value for prefix in benches for key, value in run_metrics(records[prefix], subsets[prefix], prefix).items()}
if MATCHING != "weighted-ncc":
    for prefix in benches:
        metrics.update(paired_vs_reference(records[prefix], subsets[prefix], prefix))
{k: round(v, 3) for k, v in metrics.items() if "_at_" not in k}

In [ ]:
pd.concat({prefix or "primary": r.groupby(["place", "noise_m"])[["error_m", "harm", "reachable"]].mean()
           for prefix, r in records.items()}).round(3)

# Timing

In [ ]:
timing_model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, TIMING_DEVICE, SEG_ARCHITECTURE)
timing_correlator = make_correlator(PLACES[0], TIMING_DEVICE, timing_model)
metrics["ms_per_correction"] = median_ms_per_correction(timing_correlator, benches[""][PLACES[0]][:TIMING_FRAMES],
                                                        PRIMARY_NOISE_M, N_DIRECTIONS, TIMING_REPEATS)
metrics["ms_per_correction"]

# Plots

In [ ]:
def plot_error_cdf(records: pd.DataFrame, title: str) -> Figure:
    """Error CDF per injected level, the level marked as a dotted line."""
    fig, ax = plt.subplots(figsize=(6, 4))
    for noise_m, group in records.groupby("noise_m"):
        errors = np.sort(group["error_m"].to_numpy())
        line, = ax.plot(errors, np.linspace(0, 1, len(errors)), label=f"{noise_m:g} m injected")
        ax.axvline(noise_m, color=line.get_color(), ls=":", lw=1)
    ax.set_xlabel("position error after correction, m"); ax.set_ylabel("share of draws")
    ax.set_title(title); ax.legend(); ax.grid(alpha=.3)
    fig.tight_layout()
    return fig

In [ ]:
def plot_per_frame(records: pd.DataFrame, title: str) -> Figure:
    """Mean error per frame at the primary level, sorted."""
    per_frame = records[records["noise_m"] == PRIMARY_NOISE_M].groupby("frame")["error_m"].mean().sort_values()
    fig, ax = plt.subplots(figsize=(12, 3.5))
    per_frame.plot.bar(ax=ax, width=.8)
    ax.axhline(PRIMARY_NOISE_M, color="k", ls=":", lw=1)
    ax.set_ylabel(f"mean error @ {PRIMARY_NOISE_M:g} m, m"); ax.set_title(title)
    fig.tight_layout()
    return fig

In [ ]:
def plot_pmi(weights: np.ndarray, title: str) -> Figure:
    """The PMI table, drone class in rows, satellite class in columns; diverging around 0 (chance)."""
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    limit = float(np.abs(weights).max())
    image = ax.imshow(weights, cmap="RdBu_r", norm=TwoSlopeNorm(0.0, -limit, limit))
    for (row, col), value in np.ndenumerate(weights):
        ax.text(col, row, f"{value:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(value) > limit * 0.6 else "black")
    ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=45, ha="right"); ax.set_yticks(range(len(CLASSES)), CLASSES)
    ax.set_xlabel("satellite class j (map date)"); ax.set_ylabel("drone class i (current)")
    ax.set_title(title)
    fig.colorbar(image, ax=ax, label="log P(i,j) / P(i)P(j)")
    fig.tight_layout()
    return fig

In [ ]:
figures = [plot(r, f"{MATCHING}, {prefix or 'primary'} bench")
           for prefix, r in records.items() for plot in (plot_error_cdf, plot_per_frame)]
if W is not None:
    figures.append(plot_pmi(W, f"W: {len(pmi_bank)} pairs, {len(train_villages)} train villages"))

# Log

In [ ]:
run_id = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=figures,
                 seed=SEED, device=DEVICE)["run"] if LOG_RUN else "dry"
(artifacts / "split.json").write_text(json.dumps({"train": train_villages, "val": val_villages}, indent=1))
for prefix, r in records.items():
    r.to_csv(artifacts / f"{prefix}records-{run_id}.csv", index=False)
if W is not None:
    (experiment_dir(EXP) / "figures").mkdir(exist_ok=True)
    figures[-1].savefig(experiment_dir(EXP) / "figures" / "pmi-W.png", dpi=150)
run_id